# Importing Libraries

In [114]:
import pandas as pd
import requests
from pathlib import Path
from bs4 import BeautifulSoup
from datetime import datetime, timezone, timedelta
import json
import re
from html import unescape

# Setup Configurations

In [115]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
COMPANIES_PATH = DATA_DIR / "companies.csv"

OUTPUT_PATH = BASE_DIR / "data" / "jobs_raw.csv"



print(f"Base directory: {BASE_DIR} | Base directory exists: {BASE_DIR.exists()}")
print(f"Companies path: {COMPANIES_PATH} |File exists: {COMPANIES_PATH.exists()}")
print(f"Output Path: {OUTPUT_PATH} | Output directory exists: {OUTPUT_PATH.parent.exists()}")

Base directory: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai | Base directory exists: True
Companies path: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai\data\companies.csv |File exists: True
Output Path: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai\data\jobs_raw.csv | Output directory exists: True


## Load the registry

In [116]:
companies_df = pd.read_csv(COMPANIES_PATH)
companies_df.head()

,company,career_url,ats_type,ats_slug,workday_tenant,workday_site,workday_server,keywords,locations,priority
0,OpenAI,https://openai.com/careers/search/,ashby,openai,NaN,NaN,NaN,machine learning;data scientist;ai engineer;so...,United States;Remote,high
1,Anthropic,https://www.anthropic.com/jobs,greenhouse,anthropic,NaN,NaN,NaN,machine learning;data scientist;ai engineer;so...,United States;Remote,high
2,Netflix,https://explore.jobs.netflix.net/careers,custom_netflix,NaN,NaN,NaN,NaN,machine learning;data scientist;software engineer,United States;Remote,medium
3,Workday,https://workday.wd5.myworkdayjobs.com/Workday,workday,NaN,workday,Workday,wd5,machine learning;data scientist;software engineer,United States;Remote,medium
4,Spotify,https://www.lifeatspotify.com/jobs,lever,spotify,NaN,NaN,NaN,machine learning;data scientist;software engin...,United States;Remote,high


# Helper Functions

In [117]:
Headers ={
    "User-Agent": "Mozilla/5.0 JobScout/0.1"
}

In [118]:
def clean_html(html_text):
    """
    Converts raw HTML into readable plain text.
    Works for Greenhouse, Ashby, Lever, and Workday descriptions.
    """
    if html_text is None:
        return ""

    html_text = unescape(str(html_text))

    soup = BeautifulSoup(html_text, "html.parser")

    # Remove scripts/styles
    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text(separator=" ")

    # Normalize whitespace
    text = " ".join(text.split())

    return text

## normalize tags

In [119]:
def first_available(data, keys, default=""):
    """
    Return the first non-empty value from a dictionary.
    """
    for key in keys:
        value = data.get(key)
        if value not in [None, "", [], {}]:
            return value
    return default



In [120]:

def normalize_location(value):
    """
    Convert different location formats into a readable string.
    """
    if not value:
        return ""

    if isinstance(value, str):
        return value

    if isinstance(value, dict):
        return first_available(value, ["name", "location", "city", "text"])

    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(first_available(item, ["name", "location", "city", "text"]))
        return ", ".join([p for p in parts if p])

    return str(value)

## Greenhouse

In [121]:
def fetch_greenhouse(company, ats_slug):
    url = f"https://boards-api.greenhouse.io/v1/boards/{ats_slug}/jobs?content=true"

    response = requests.get(url, headers=Headers, timeout=20)
    response.raise_for_status()

    data = response.json()
    jobs = []

    for job in data.get("jobs", []):
        location = normalize_location(
            first_available(job, [
                "location",
                "offices"
            ])
        )

        raw_description = first_available(job, ["content"])

        posted_date = first_available(job, [
            "first_published",
            "published_at",
            "updated_at",
            "created_at"
        ])

        jobs.append({
            "company": company,
            "title": first_available(job, ["title"]),
            "location": location,
            "job_url": first_available(job, ["absolute_url"]),
            "description": clean_html(raw_description),
            "description_raw": raw_description,
            "ats_type": "greenhouse",
            "external_job_id": str(first_available(job, ["id"])),
            "posted_date": posted_date,
            "date_found": datetime.now(timezone.utc).isoformat()
        })

    return jobs

## Ashby fetcher

In [122]:
def fetch_ashby(company, ats_slug):
    url = f"https://api.ashbyhq.com/posting-api/job-board/{ats_slug}?includeCompensation=true"

    response = requests.get(url, headers=Headers, timeout=20)
    response.raise_for_status()

    data = response.json()
    jobs = []

    for job in data.get("jobs", []):
        location = normalize_location(
            first_available(job, [
                "location",
                "locations",
                "locationName",
                "address",
                "office",
                "offices"
            ])
        )

        description = first_available(job, [
            "descriptionHtml",
            "descriptionPlain",
            "description",
            "jobDescription"
        ])

        posted_date = first_available(job, [
            "publishedAt",
            "publishedDate",
            "postedDate",
            "createdAt",
            "updatedAt"
        ])

        jobs.append({
            "company": company,
            "title": first_available(job, ["title", "name"]),
            "location": location,
            "job_url": first_available(job, ["jobUrl", "applyUrl", "url"]),
            "description": clean_html(description),
            "ats_type": "ashby",
            "external_job_id": str(first_available(job, ["id", "jobId"])),
            "posted_date": posted_date,
            "date_found": datetime.now(timezone.utc).isoformat()
        })

    return jobs

## Netflix Fetcher

In [123]:
def extract_json_array_after_key(text, key):
    """
    Extracts a JSON array after a key like "positions": [...]
    This is safer than regex for huge nested JSON.
    """
    key_pattern = f'"{key}"'
    key_index = text.find(key_pattern)

    if key_index == -1:
        return None

    array_start = text.find("[", key_index)

    if array_start == -1:
        return None

    bracket_count = 0
    in_string = False
    escape = False

    for i in range(array_start, len(text)):
        char = text[i]

        if escape:
            escape = False
            continue

        if char == "\\":
            escape = True
            continue

        if char == '"':
            in_string = not in_string

        if not in_string:
            if char == "[":
                bracket_count += 1
            elif char == "]":
                bracket_count -= 1

                if bracket_count == 0:
                    return text[array_start:i + 1]

    return None

In [124]:
def fetch_netflix(company, career_url):
    response = requests.get(career_url, headers=Headers, timeout=20)
    response.raise_for_status()

    html = response.text

    positions_json = extract_json_array_after_key(html, "positions")

    if not positions_json:
        print("Could not find Netflix positions data.")
        return []

    positions = json.loads(positions_json)

    jobs = []

    for job in positions:
        created_ts = job.get("t_create")
        updated_ts = job.get("t_update")

        posted_date = (
            datetime.fromtimestamp(created_ts, timezone.utc).isoformat()
            if created_ts else ""
        )

        updated_date = (
            datetime.fromtimestamp(updated_ts, timezone.utc).isoformat()
            if updated_ts else ""
        )

        jobs.append({
            "company": company,
            "title": job.get("posting_name") or job.get("name", ""),
            "location": normalize_location(job.get("locations") or job.get("location")),
            "job_url": job.get("canonicalPositionUrl", ""),
            "description": clean_html(job.get("job_description", "")),
            "ats_type": "custom_netflix",
            "external_job_id": str(job.get("ats_job_id") or job.get("id", "")),
            "posted_date": posted_date,
            "updated_date": updated_date,
            "department": job.get("department", ""),
            "business_unit": job.get("business_unit", ""),
            "work_location_option": job.get("work_location_option", ""),
            "date_found": datetime.now(timezone.utc).isoformat()
        })

    return jobs

## Lever

In [125]:
def extract_lever_description(job):
    description_parts = []

    # Main description
    description_parts.append(first_available(job, [
        "descriptionPlain",
        "description",
    ]))

    # Additional section
    description_parts.append(first_available(job, [
        "additionalPlain",
        "additional",
    ]))

    # Important: Lever stores responsibilities/requirements here
    for section in job.get("lists", []):
        section_title = section.get("text", "")
        section_content = clean_html(section.get("content", ""))

        if section_title or section_content:
            description_parts.append(f"{section_title}: {section_content}")

    return " ".join([part for part in description_parts if part])

In [126]:
def fetch_lever(company, ats_slug):
    url = f"https://api.lever.co/v0/postings/{ats_slug}?mode=json"

    response = requests.get(url, headers=Headers, timeout=20)
    response.raise_for_status()

    data = response.json()
    jobs = []

    for job in data:
        categories = job.get("categories", {}) or {}

        location_parts = [
            first_available(categories, ["location", "team", "department"]),
            first_available(job, ["country"]),
            first_available(job, ["workplaceType"]),
        ]
        location = " ".join(str(part) for part in location_parts if part)

        description = extract_lever_description(job)

        posted_date = first_available(job, [
            "createdAt",
            "updatedAt",
            "created_at",
            "updated_at"
        ])

        jobs.append({
            "company": company,
            "title": first_available(job, ["text", "title"]),
            "location": location,
            "job_url": first_available(job, ["hostedUrl", "applyUrl", "url"]),
            "description": description,
            "ats_type": "lever",
            "external_job_id": str(first_available(job, ["id"])),
            "posted_date": posted_date,
            "date_found": datetime.now(timezone.utc).isoformat()
        })

    return jobs

## Workday

In [127]:
def fetch_workday_detail(base_url, tenant, site, external_path):
    """
    Fetches full Workday job detail using the externalPath from the list API.
    """
    if not external_path:
        return {}

    detail_url = f"{base_url}/wday/cxs/{tenant}/{site}{external_path}"

    response = requests.get(detail_url, headers=Headers, timeout=20)
    response.raise_for_status()

    return response.json()

In [128]:
def fetch_workday(company, tenant, site, server):
    base_url = f"https://{tenant}.{server}.myworkdayjobs.com"
    endpoint = f"{base_url}/wday/cxs/{tenant}/{site}/jobs"

    payload = {
        "appliedFacets": {},
        "limit": 20,
        "offset": 0,
        "searchText": ""
    }

    response = requests.post(endpoint, json=payload, headers=Headers, timeout=20)
    response.raise_for_status()

    data = response.json()
    postings = data.get("jobPostings", [])

    jobs = []

    for job in postings:
        external_path = first_available(job, ["externalPath"])
        job_url = base_url + external_path if external_path.startswith("/") else external_path

        # List-level fields
        title = first_available(job, ["title"])
        locations = normalize_location(
            first_available(job, [
                "locationsText",
                "locations",
                "location"
            ])
        )

        posted_date = first_available(job, [
            "postedOn",
            "postedDate",
            "startDate",
            "createdAt",
            "updatedAt"
        ])

        remote_type = first_available(job, ["remoteType"])
        requisition_id = ""

        bullet_fields = job.get("bulletFields", [])
        if isinstance(bullet_fields, list) and bullet_fields:
            requisition_id = str(bullet_fields[0])

        # Detail-level fields
        try:
            detail_data = fetch_workday_detail(base_url, tenant, site, external_path)
        except Exception as e:
            print(f"Could not fetch Workday detail for {title}: {e}")
            detail_data = {}

        job_info = detail_data.get("jobPostingInfo", {}) or detail_data

        raw_description = first_available(job_info, [
            "jobDescription",
            "description",
            "jobDescriptionText"
        ])

        # Sometimes title/location/posting date are better in detail response
        title = first_available(job_info, ["title"], default=title)

        location = normalize_location(
            first_available(job_info, [
                "location",
                "locations",
                "locationsText"
            ], default=locations)
        )

        posted_date = first_available(job_info, [
            "postedOn",
            "postedDate",
            "startDate"
        ], default=posted_date)

        requisition_id = first_available(job_info, [
            "jobReqId",
            "jobRequisitionId",
            "requisitionId"
        ], default=requisition_id)


        jobs.append({
            "company": company,
            "title": title,
            "location": location,
            "job_url": job_url,
            "description": clean_html(raw_description),
            "description_raw": raw_description,
            "ats_type": "workday",
            "external_job_id": str(requisition_id or external_path),
            "posted_date": posted_date,
            "remote_type": remote_type,
            "date_found": datetime.now(timezone.utc).isoformat()
        })

    return jobs

## Route by ATS

In [129]:
def fetch_jobs_for_company(row):
    ats_type = row["ats_type"].lower().strip()

    if ats_type == "greenhouse":
        return fetch_greenhouse(row["company"], row["ats_slug"])

    if ats_type == "ashby":
        return fetch_ashby(row["company"], row["ats_slug"])

    if ats_type == "lever":
        return fetch_lever(row["company"], row["ats_slug"])

    if ats_type == "workday":
        return fetch_workday(
            row["company"],
            row["workday_tenant"],
            row["workday_site"],
            row["workday_server"]
        )
    if ats_type == "custom_netflix":
        return fetch_netflix(row["company"], row["career_url"])

    print(f"Skipping {row['company']} because ATS type '{ats_type}' is not supported yet.")
    return []

## Normalize Posted date

In [130]:
def parse_posted_date(value):
    """
    Converts messy ATS posting dates into UTC datetime objects.
    Handles:
    - ISO dates
    - Unix timestamps
    - Workday strings like "Posted Today", "Posted Yesterday", "Posted 5 Days Ago"
    """

    if value is None:
        return None

    if pd.isna(value):
        return None

    value = str(value).strip()

    if value == "":
        return None

    value_lower = value.lower()
    now = datetime.now(timezone.utc)

    # Workday-style relative dates
    if "posted today" in value_lower or value_lower == "today":
        return now

    if "posted yesterday" in value_lower or value_lower == "yesterday":
        return now - timedelta(days=1)

    match = re.search(r"posted\s+(\d+)\s+days?\s+ago", value_lower)
    if match:
        days = int(match.group(1))
        return now - timedelta(days=days)

    if "30+ days" in value_lower:
        return now - timedelta(days=30)

    # Unix timestamp: seconds or milliseconds
    if value.isdigit():
        timestamp = int(value)

        if timestamp > 10**12:
            timestamp = timestamp / 1000

        try:
            return datetime.fromtimestamp(timestamp, timezone.utc)
        except Exception:
            return None

    # ISO/date string fallback
    parsed = pd.to_datetime(value, utc=True, errors="coerce")

    if pd.isna(parsed):
        return None

    return parsed.to_pydatetime()


## Classify Freshness

In [131]:
def classify_freshness(posted_datetime, hours=24):
    """
    Labels jobs as:
    - fresh: posted within last N hours
    - old: older than N hours
    - unknown: no parseable posted date
    """
    now = datetime.now(timezone.utc)

    if posted_datetime is None or pd.isna(posted_datetime):
        return "unknown"

    age = now - posted_datetime

    if age <= timedelta(hours=hours):
        return "fresh"

    return "old"

## End

# Run Fetch

In [132]:
all_jobs = []

for _, row in companies_df.iterrows():
    print(f"Fetching jobs for {row['company']}...")

    try:
        jobs = fetch_jobs_for_company(row)
        print(f"Found {len(jobs)} jobs")
        all_jobs.extend(jobs)
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error for {row['company']}: {e}")
        print("Response text:", e.response.text[:500])
    except Exception as e:
        print(f"Error fetching {row['company']}: {e}")

jobs_df = pd.DataFrame(all_jobs)


Fetching jobs for OpenAI...
Found 713 jobs
Fetching jobs for Anthropic...
Found 366 jobs
Fetching jobs for Netflix...
Could not find Netflix positions data.
Found 0 jobs
Fetching jobs for Workday...
Found 20 jobs
Fetching jobs for Spotify...
Found 166 jobs


In [133]:
jobs_df[jobs_df['ats_type'] == "ashby"].head()

,company,title,location,job_url,description,ats_type,external_job_id,posted_date,date_found,description_raw,remote_type
0,OpenAI,"Technical Program Manager, Compute Infrastructure",San Francisco,https://jobs.ashbyhq.com/openai/8fb1615c-34bf-...,About the Team The compute infrastructure team...,ashby,8fb1615c-34bf-47c4-a1d1-b7b2f836bbd3,2026-03-12T16:38:15.322+00:00,2026-06-04T18:11:09.402369+00:00,NaN,NaN
1,OpenAI,Research Engineer,San Francisco,https://jobs.ashbyhq.com/openai/240d459b-696d-...,"By applying to this role, you will be consider...",ashby,240d459b-696d-43eb-8497-fab3e56ecd9b,2025-04-05T00:03:20.653+00:00,2026-06-04T18:11:09.407935+00:00,NaN,NaN
2,OpenAI,Account Director - Tokyo,"Tokyo, Japan",https://jobs.ashbyhq.com/openai/18f58952-c242-...,About the team OpenAI’s mission is to build sa...,ashby,18f58952-c242-4562-8732-073a0ae8029e,2026-01-23T00:17:18.483+00:00,2026-06-04T18:11:09.410524+00:00,NaN,NaN
3,OpenAI,"Software Engineer, RL Training Infra",San Francisco,https://jobs.ashbyhq.com/openai/13995549-e8cc-...,About the Team The Post-Training Frontiers tea...,ashby,13995549-e8cc-498f-9eaa-1869067ac35b,2026-05-23T02:00:50.464+00:00,2026-06-04T18:11:09.410524+00:00,NaN,NaN
4,OpenAI,"Research Engineer, Retrieval & Search, Applied...",San Francisco,https://jobs.ashbyhq.com/openai/7322d344-9325-...,About the Team We bring OpenAI's technology to...,ashby,7322d344-9325-4a92-8445-0a2c4e9272f8,2024-03-20T21:33:20.763+00:00,2026-06-04T18:11:09.410524+00:00,NaN,NaN


In [134]:

# jobs_df.to_csv(OUTPUT_PATH, index=False)
# print("Saved jobs to:", OUTPUT_PATH)
# print("Total jobs:", len(jobs_df))

# Inspect data quality

In [135]:
jobs_df.head()

,company,title,location,job_url,description,ats_type,external_job_id,posted_date,date_found,description_raw,remote_type
0,OpenAI,"Technical Program Manager, Compute Infrastructure",San Francisco,https://jobs.ashbyhq.com/openai/8fb1615c-34bf-...,About the Team The compute infrastructure team...,ashby,8fb1615c-34bf-47c4-a1d1-b7b2f836bbd3,2026-03-12T16:38:15.322+00:00,2026-06-04T18:11:09.402369+00:00,NaN,NaN
1,OpenAI,Research Engineer,San Francisco,https://jobs.ashbyhq.com/openai/240d459b-696d-...,"By applying to this role, you will be consider...",ashby,240d459b-696d-43eb-8497-fab3e56ecd9b,2025-04-05T00:03:20.653+00:00,2026-06-04T18:11:09.407935+00:00,NaN,NaN
2,OpenAI,Account Director - Tokyo,"Tokyo, Japan",https://jobs.ashbyhq.com/openai/18f58952-c242-...,About the team OpenAI’s mission is to build sa...,ashby,18f58952-c242-4562-8732-073a0ae8029e,2026-01-23T00:17:18.483+00:00,2026-06-04T18:11:09.410524+00:00,NaN,NaN
3,OpenAI,"Software Engineer, RL Training Infra",San Francisco,https://jobs.ashbyhq.com/openai/13995549-e8cc-...,About the Team The Post-Training Frontiers tea...,ashby,13995549-e8cc-498f-9eaa-1869067ac35b,2026-05-23T02:00:50.464+00:00,2026-06-04T18:11:09.410524+00:00,NaN,NaN
4,OpenAI,"Research Engineer, Retrieval & Search, Applied...",San Francisco,https://jobs.ashbyhq.com/openai/7322d344-9325-...,About the Team We bring OpenAI's technology to...,ashby,7322d344-9325-4a92-8445-0a2c4e9272f8,2024-03-20T21:33:20.763+00:00,2026-06-04T18:11:09.410524+00:00,NaN,NaN


## Missing description or posted date

In [136]:
print("Total jobs:", len(jobs_df))

jobs_df.groupby("ats_type").agg(
    job_count=("title", "count"),
    descriptions_present=("description", lambda x: (x.fillna("").str.len() > 0).sum()),
    posted_dates_present=("posted_date", lambda x: (x.fillna("") != "").sum())
)

Total jobs: 1265


C:\Users\surab\AppData\Local\Temp\ipykernel_25988\696228565.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  posted_dates_present=("posted_date", lambda x: (x.fillna("") != "").sum())


,job_count,descriptions_present,posted_dates_present
ats_type,,,
ashby,713,713,713
greenhouse,366,366,366
lever,166,166,166
workday,20,20,20


## Normalize posted dates

In [137]:
jobs_df["posted_datetime"] = jobs_df["posted_date"].apply(parse_posted_date)



In [138]:
jobs_df[["company", "title", "ats_type", "posted_date", "posted_datetime"]].query("company == 'Workday'").head(20)

,company,title,ats_type,posted_date,posted_datetime
1079,Workday,Senior Software Engineer (Gen AI),workday,Posted Today,2026-06-04 18:16:01.931121+00:00
1080,Workday,"Principal Product Advisor, Revenue Management",workday,Posted Today,2026-06-04 18:16:01.931121+00:00
1081,Workday,"Principal Product Advisor, Financials",workday,Posted Today,2026-06-04 18:16:01.931121+00:00
1082,Workday,"Principal Product Advisor, Spend Management",workday,Posted Today,2026-06-04 18:16:01.931121+00:00
1083,Workday,Medium Enterprise Regional Sales Director - Lo...,workday,Posted Today,2026-06-04 18:16:01.931121+00:00
1084,Workday,Principal Program Manager - Finance,workday,Posted Today,2026-06-04 18:16:01.931121+00:00
1085,Workday,Sr. Sourcing Manager,workday,Posted Today,2026-06-04 18:16:01.931121+00:00
1086,Workday,Account Executive - Large Enterprise UKI,workday,Posted Today,2026-06-04 18:16:01.931121+00:00
1087,Workday,"Sr Manager, Software Development Engineering -...",workday,Posted Today,2026-06-04 18:16:01.931121+00:00
1088,Workday,Account Executive - Customer Base LE,workday,Posted Today,2026-06-04 18:16:01.931121+00:00


## Check posted date coverage

In [139]:
jobs_df.groupby("ats_type").agg(
    total_jobs=("title", "count"),
    parsed_dates=("posted_datetime", lambda x: x.notna().sum())
)

,total_jobs,parsed_dates
ats_type,,
ashby,713,713
greenhouse,366,366
lever,166,166
workday,20,20


## Freshness

In [140]:
jobs_df["freshness_status"] = jobs_df["posted_datetime"].apply(
    lambda x: classify_freshness(x, hours=24)
)



In [141]:
jobs_df["freshness_status"].value_counts()

freshness_status
old      1236
fresh      29
Name: count, dtype: int64

In [142]:
jobs_df.groupby("ats_type").agg(
    total_jobs=("title", "count"),
    parsed_dates=("posted_datetime", lambda x: x.notna().sum()),
    fresh_jobs=("freshness_status", lambda x: (x == "fresh").sum()),
    unknown_dates=("freshness_status", lambda x: (x == "unknown").sum()),
    old_jobs=("freshness_status", lambda x: (x == "old").sum())
)

,total_jobs,parsed_dates,fresh_jobs,unknown_dates,old_jobs
ats_type,,,,,
ashby,713,713,11,0,702
greenhouse,366,366,5,0,361
lever,166,166,0,0,166
workday,20,20,13,0,7


In [144]:
fresh_or_unknown_jobs_df = jobs_df[
    jobs_df["freshness_status"].isin(["fresh", "unknown"])
].copy()

print("Jobs after freshness filter:", len(fresh_or_unknown_jobs_df))

Jobs after freshness filter: 29


# Save CSV

In [146]:


jobs_df.to_csv(OUTPUT_PATH, index=False)

print("Saved raw jobs to:", OUTPUT_PATH)
print("Total jobs:", len(jobs_df))
print(jobs_df["freshness_status"].value_counts())

Saved raw jobs to: c:\Users\surab\OneDrive\Documents\Personal\Projects\jobscout-ai\data\jobs_raw.csv
Total jobs: 1265
freshness_status
old      1236
fresh      29
Name: count, dtype: int64


## End